In [1]:
import requests
import json
from datetime import datetime
import time
import random
import asyncio, aiohttp
import aiolimiter


In [2]:
_TOKEN_CACHE = None  # global cache

def get_session_token():
    global _TOKEN_CACHE
    if _TOKEN_CACHE:   # reuse valid token
        return _TOKEN_CACHE

    url = "https://bsky.social/xrpc/com.atproto.server.createSession"
    handle = "repostproj.bsky.social"
    app_password = "vyvc-xg5q-seda-utaz" 

    for _ in range(3):  # retry a few times on rate limit
        r = requests.post(url, json={"identifier": handle, "password": app_password})
        if r.status_code == 429:
            print("⚠️ Rate-limited on login, sleeping 5s...")
            time.sleep(1)
            continue
        r.raise_for_status()
        _TOKEN_CACHE = r.json()["accessJwt"]
        return _TOKEN_CACHE

    raise RuntimeError("Failed to obtain token after retries")

In [3]:


def load_posts_dict(file_path):
    """
    Reads a .jsonl file where each line is a JSON object representing a post.
    Returns a dict mapping post_id (or URI) -> post data.

    Supports multiple Bluesky data formats:
      - post["uri"]
      - post["post"]["uri"]
      - post["id"]
    """
    posts_dict = {}
    bad_lines = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                post = json.loads(line)
            except json.JSONDecodeError:
                bad_lines += 1
                continue

            # Try to find a reliable ID/URI key
            post_id = (
                post.get("uri")
                or post.get("post", {}).get("uri")
                or post.get("id")
            )

            posts_dict[post_id] = post

    print(f"✅ Loaded {len(posts_dict)} posts from {file_path} ({bad_lines} bad lines skipped)")
    return posts_dict


In [8]:
posts_dict = load_posts_dict("test_30000_trump.jsonl")   

✅ Loaded 64765 posts from test_30000_trump.jsonl (0 bad lines skipped)


In [14]:


API_URL = "https://public.api.bsky.app/xrpc/app.bsky.feed.getRepostedBy"


async def fetch_reposters(session, uri, limit=100):
    """Fetch reposters for one post URI safely."""
    params = {"uri": uri, "limit": limit}
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        async with session.get(API_URL, params=params, headers=headers, timeout=300) as r:
            if r.status != 200:
                print(f"HTTP {r.status} for {uri}")
                return []
            data = await r.json()
            return [u["did"] for u in data.get("repostedBy", [])]
    except Exception as e:
        print(f"Error fetching {uri}: {e}")
        return []


async def collect_reposter_dict(posts_dict, concurrency=25):
    """
    Takes a dict of posts:
        { post_uri: {...}, post_uri2: {...}, ... }

    Returns a dict mapping each reposter DID -> [list of post URIs they reposted].
    Only includes posts that have repostCount > 0.
    """
    posts_to_check, tasks = [], []

    connector = aiohttp.TCPConnector(limit_per_host=concurrency)
    async with aiohttp.ClientSession(connector=connector) as session:

        for post_uri, post in posts_dict.items():
            repost_count = post.get("repostCount")

            if repost_count > 0:
                posts_to_check.append(post_uri)
                tasks.append(fetch_reposters(session, post_uri))

        # Run all requests concurrently
        repost_lists = await asyncio.gather(*tasks, return_exceptions=True)

    # Merge results into reposter → [post_uris] mapping
    reposter_dict = {}
    for post_uri, reposters in zip(posts_to_check, repost_lists):
        if not reposters or isinstance(reposters, Exception):
            continue
        for reposter in reposters:
            reposter_dict.setdefault(reposter, []).append(post_uri)

    print(f"Processed {len(posts_to_check)} posts. Found {len(reposter_dict)} unique reposters.")
    return reposter_dict


In [192]:
repost_dict = await collect_reposter_dict(posts_dict)

⚠️ HTTP 500 for at://did:plc:siis77xqtibffyigtyrrr5ij/app.bsky.feed.post/3m3qiot6uc22q
⚠️ HTTP 500 for at://did:plc:jfjfl4hw3dm4zrxglniwkjnf/app.bsky.feed.post/3m3b27b4fu332
✅ Processed 14760 posts. Found 37433 unique reposters.


In [15]:
print(len(repost_dict))

37433


In [16]:


FOLLOW_API = "https://public.api.bsky.app/xrpc/app.bsky.graph.getFollows"


def get_author_dids(posts_dict):
    """Extract all unique author DIDs from post data."""
    authors = set()
    for post in posts_dict.values():
        did = (
            post.get("author", {}).get("did")
            or post.get("post", {}).get("author", {}).get("did")
        )
        if did:
            authors.add(did)
    return authors


async def fetch_follows(session, reposter, author_set, retries=3, first_page_only=True):
    """Fetch follows for one reposter; retry on transient failures."""
    follows = set()
    cursor = None
    headers = {"User-Agent": "Mozilla/5.0"}

    delay = 1
    for attempt in range(retries):
        try:
            while True:
                params = {"actor": reposter, "limit": 100}
                if cursor:
                    params["cursor"] = cursor

                async with session.get(FOLLOW_API, params=params, headers=headers) as r:
                    if r.status != 200:
                        if 500 <= r.status < 600 and attempt < retries - 1:
                            await asyncio.sleep(delay)
                            delay *= 2
                            continue
                        print(f"⚠️ HTTP {r.status} for {reposter}")
                        return reposter, []

                    data = await r.json()
                    follows.update(
                        u["did"] for u in data.get("follows", [])
                        if u.get("did") in author_set
                    )

                    cursor = data.get("cursor")
                    if not cursor or first_page_only:
                        break
            return reposter, list(follows)

        except Exception as e:
            if attempt < retries - 1:
                await asyncio.sleep(delay)
                delay *= 2
                continue
            print(f" {reposter} failed after {retries} retries: {e}")
            return reposter, []


async def collect_reposter_followed_authors(reposter_dict, posts_dict,
                                            concurrency=100, reqs_per_sec=50):
    """
    Fetch who each reposter follows (intersected with author_set),
    concurrently and rate-limited for stability.
    """
    reposters = list(reposter_dict.keys())
    total = len(reposters)
    author_set = get_author_dids(posts_dict)
    counter = {"done": 0}

    # --- tuned network setup ---
    rate_limiter = aiolimiter.AsyncLimiter(reqs_per_sec, 1)  # 50 req/s max
    connector = aiohttp.TCPConnector(limit=300, limit_per_host=50, ttl_dns_cache=300)
    timeout = aiohttp.ClientTimeout(total=10, connect=5, sock_read=5)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        sem = asyncio.Semaphore(concurrency)

        async def limited_fetch(reposter):
            async with sem, rate_limiter:
                rep, authors = await fetch_follows(session, reposter, author_set)
                counter["done"] += 1
                if counter["done"] % 100 == 0 or counter["done"] == total:
                    print(f"Progress: {counter['done']}/{total} "
                          f"({counter['done']/total:.1%})")
                return rep, authors

        tasks = [limited_fetch(r) for r in reposters]
        responses = await asyncio.gather(*tasks)

    return {r: authors for r, authors in responses if authors}


In [219]:
followers = await collect_reposter_followed_authors(repost_dict,posts_dict)

Progress: 100/37433 (0.3%)
Progress: 200/37433 (0.5%)
Progress: 300/37433 (0.8%)
Progress: 400/37433 (1.1%)
Progress: 500/37433 (1.3%)
Progress: 600/37433 (1.6%)
Progress: 700/37433 (1.9%)
Progress: 800/37433 (2.1%)
Progress: 900/37433 (2.4%)
Progress: 1000/37433 (2.7%)
Progress: 1100/37433 (2.9%)
Progress: 1200/37433 (3.2%)
Progress: 1300/37433 (3.5%)
Progress: 1400/37433 (3.7%)
Progress: 1500/37433 (4.0%)
Progress: 1600/37433 (4.3%)
Progress: 1700/37433 (4.5%)
Progress: 1800/37433 (4.8%)
Progress: 1900/37433 (5.1%)
Progress: 2000/37433 (5.3%)
Progress: 2100/37433 (5.6%)
Progress: 2200/37433 (5.9%)
Progress: 2300/37433 (6.1%)
Progress: 2400/37433 (6.4%)
Progress: 2500/37433 (6.7%)
Progress: 2600/37433 (6.9%)
Progress: 2700/37433 (7.2%)
Progress: 2800/37433 (7.5%)
Progress: 2900/37433 (7.7%)
Progress: 3000/37433 (8.0%)
Progress: 3100/37433 (8.3%)
Progress: 3200/37433 (8.5%)
Progress: 3300/37433 (8.8%)
Progress: 3400/37433 (9.1%)
Progress: 3500/37433 (9.4%)
Progress: 3600/37433 (9.6%)
P

In [17]:
def find_negative_instances(reposter_did, follow_map, posts_dict, reposter_dict, num_of_instances):
    """
    Returns (post_uri, reposter_did) if there's a post whose author is followed
    by the reposter, but that post was NOT reposted by them.
    """
    followed_authors = set(follow_map.get(reposter_did, []))
    reposted_posts = set(reposter_dict.get(reposter_did, []))

    instances = []
    for post_uri, post in posts_dict.items():
        if num_of_instances == 0: break
        author = (
            post.get("author", {}).get("did")
            or post.get("post", {}).get("author", {}).get("did")
        )
        if not author or author not in followed_authors:
            continue
        if post_uri not in reposted_posts:
            instances.append(post_uri)
            num_of_instances-=1
        
    return instances


In [224]:
count=0
for i in range(len(repost_dict)):
    reposter = list(repost_dict.keys())[i]
    neg = find_negative_instances(reposter,followers,posts_dict,repost_dict,5)
    if len(neg) < 5:
        count+=1
print((len(repost_dict)-count)*6)

207576


In [19]:
with open("posts_dict.json", "w", encoding="utf-8") as f:
    json.dump(posts_dict, f, ensure_ascii=False, indent=2)

In [ ]:


with open("reposts_dict.json", "w", encoding="utf-8") as f:
    json.dump(repost_dict, f, ensure_ascii=False, indent=2)

with open("followers_dict.json", "w", encoding="utf-8") as f:
    json.dump(followers, f, ensure_ascii=False, indent=2)


NameError: name 'posts_dict' is not defined

In [18]:


async def collect_reposter_repost_times(repost_dict, concurrency=300, reqs_per_sec=100):
    """
    ⚡ Ultra-fast concurrent scraper of repost timestamps (reason.indexedAt)
    using public Bluesky AppView API (no token).
    Tuned for high throughput (~80–100 req/s sustained).
    """

    reposters = list(repost_dict.keys())
    total = len(reposters)
    counter = {"done": 0}
    start_time = time.time()

    base_url = "https://public.api.bsky.app/xrpc/app.bsky.feed.getAuthorFeed"
    rate_limiter = aiolimiter.AsyncLimiter(reqs_per_sec, 1)
    connector = aiohttp.TCPConnector(limit=600, limit_per_host=300, ttl_dns_cache=600)
    timeout = aiohttp.ClientTimeout(total=8, connect=4, sock_read=4)

    async def fetch_reposts(session, reposter, target_uris):
        found, remaining = {}, set(target_uris)
        cursor, backoff = None, 1.0
        pages = 0

        while remaining:
            params = {
                "actor": reposter,
                "limit": 100,
                "filter": "posts_and_author_threads",
            }
            if cursor:
                params["cursor"] = cursor

            await asyncio.sleep(random.uniform(0, 0.01))  # micro jitter

            try:
                async with session.get(base_url, params=params) as resp:
                    if resp.status == 429:
                        wait = backoff + random.uniform(0.05, 0.2)
                        await asyncio.sleep(wait)
                        backoff = min(backoff * 1.6, 6)
                        continue
                    if resp.status != 200:
                        break
                    data = await resp.json()
            except Exception:
                await asyncio.sleep(backoff)
                backoff = min(backoff * 1.6, 6)
                continue

            pages += 1
            feed = data.get("feed", [])
            if not feed:
                break

            for item in feed:
                reason = item.get("reason")
                post = item.get("post", {})
                if not reason or reason.get("$type") != "app.bsky.feed.defs#reasonRepost":
                    continue
                uri = post.get("uri")
                if uri in remaining:
                    found[uri] = reason.get("indexedAt")
                    remaining.remove(uri)

            cursor = data.get("cursor")
            if not cursor or not remaining:
                break

            # ⚡ stop early after 3 pages (heuristic)
            if pages >= 10:
                break

        return reposter, found

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        sem = asyncio.Semaphore(concurrency)

        async def limited_fetch(reposter):
            async with sem, rate_limiter:
                rep, reposts = await fetch_reposts(session, reposter, repost_dict[reposter])
                counter["done"] += 1
                if counter["done"] % 50 == 0 or counter["done"] == total:
                    elapsed = time.time() - start_time
                    rate = counter["done"] / elapsed if elapsed > 0 else 0
                    eta = (total - counter["done"]) / rate if rate > 0 else 0
                    print(f"[{counter['done']:>5}/{total}] {counter['done']/total:>6.1%} "
                          f"| {rate:>6.2f} u/s | ETA {eta/60:>5.1f} min")
                return rep, reposts

        tasks = [limited_fetch(r) for r in reposters]
        responses = await asyncio.gather(*tasks, return_exceptions=False)

    elapsed = time.time() - start_time
    print(f"✅ Done {total} users in {elapsed:.1f}s "
          f"({total/elapsed:.2f} users/s avg)")
    return {r: reposts for r, reposts in responses if reposts}


In [324]:
results = await collect_reposter_repost_times(repost_dict)


[   50/37433]   0.1% |   8.06 u/s | ETA  77.3 min
[  100/37433]   0.3% |   7.06 u/s | ETA  88.1 min
[  150/37433]   0.4% |   7.66 u/s | ETA  81.2 min
[  200/37433]   0.5% |   8.85 u/s | ETA  70.1 min
[  250/37433]   0.7% |   9.62 u/s | ETA  64.4 min
[  300/37433]   0.8% |  10.04 u/s | ETA  61.7 min
[  350/37433]   0.9% |  10.53 u/s | ETA  58.7 min
[  400/37433]   1.1% |  10.62 u/s | ETA  58.1 min
[  450/37433]   1.2% |  11.24 u/s | ETA  54.8 min
[  500/37433]   1.3% |  11.32 u/s | ETA  54.4 min
[  550/37433]   1.5% |  11.40 u/s | ETA  53.9 min
[  600/37433]   1.6% |  11.60 u/s | ETA  52.9 min
[  650/37433]   1.7% |  11.75 u/s | ETA  52.2 min
[  700/37433]   1.9% |  11.76 u/s | ETA  52.0 min
[  750/37433]   2.0% |  11.92 u/s | ETA  51.3 min
[  800/37433]   2.1% |  12.13 u/s | ETA  50.3 min
[  850/37433]   2.3% |  12.24 u/s | ETA  49.8 min
[  900/37433]   2.4% |  12.30 u/s | ETA  49.5 min
[  950/37433]   2.5% |  12.41 u/s | ETA  49.0 min
[ 1000/37433]   2.7% |  12.52 u/s | ETA  48.5 min


In [328]:

with open("repost_with_time.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

In [11]:
with open("repost_with_time.json", "r", encoding="utf-8") as f:
    repost_dict_times = json.load(f)

# Load repost_dict
with open("reposts_dict.json", "r", encoding="utf-8") as f:
    repost_dict = json.load(f)

# Load followers_dict
with open("followers_dict.json", "r", encoding="utf-8") as f:
    followers = json.load(f)

In [12]:
print(len(repost_dict["did:plc:dluwobsw5vrtkgrts7tnz7om"]))
print("\n")
print(len(repost_dict_times["did:plc:dluwobsw5vrtkgrts7tnz7om"]))

4


4


In [13]:
print(len(repost_dict), len(repost_dict_times))


37433 34159
